# Machine Learning Exercise

Now it is time to practise what you have seen in the previous notebook. Your task is to
train a model that predicts **how far a patient's diabetes will progress over one year**.

Note what kind of question that is. In week 2 you worked on the Pima Indians data, where the
question was *does this person have diabetes*, a yes or no. Here the target is a continuous
progression score, so this is a **regression**, and nothing you learned about accuracy,
precision or recall applies to it.

## Task

1. Load the data from `data/diabetes_progression.csv`.
2. Split the data into train and test.
3. Conduct a brief EDA **on the training data** to become familiar with it.
4. Preprocess as far as you need to, inside a `Pipeline`.
5. Train and tune at least two different regressors, and compare them.
6. Report the performance of your best model on the test set, once.

## What to use and keep in mind

* **Scale your data.** Which scaler works best in your case, and which models care?
* **Tune your model.** Start with a wide grid and `RandomizedSearchCV`, then narrow it with
  `GridSearchCV`.
* **Choose the right evaluation metric.** It is not accuracy. Which error is more meaningful
  here, and can you express it in the units a doctor would use?
* **Compare against a baseline.** A model is only good relative to something. What is the
  simplest possible prediction you could make?

## Data Overview

442 patients, measured at baseline, with the target measured one year later.

| column | meaning |
| :----- | :------ |
| `age` | age in years |
| `sex` | 1 or 2 |
| `bmi` | body mass index |
| `bp` | average blood pressure |
| `s1` | total serum cholesterol |
| `s2` | low-density lipoproteins, LDL |
| `s3` | high-density lipoproteins, HDL |
| `s4` | total cholesterol divided by HDL |
| `s5` | serum triglycerides, probably on a log scale |
| `s6` | blood sugar level |
| `target` | a quantitative measure of disease progression one year after baseline |

Two of those are worth knowing before you start. **HDL is the "good" cholesterol**, so `s3`
should work in the opposite direction to the others, and `s4` is a ratio built from `s1` and
`s3`, so it carries no information those two do not already have between them. Keep both in
mind when you look at correlations and at multicollinearity.

## Import

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import loguniform, uniform

# Feel free to add all the libraries you need
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, ElasticNet
from sklearn.dummy import DummyRegressor
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

RSEED = 42
sns.set_style("whitegrid")

## Getting the Data

The dataset ships with this repository at `data/diabetes_progression.csv`.

In [ ]:
df = pd.read_csv("../data/diabetes_progression.csv")

print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Check for missing values and duplicates before doing anything else
print("missing values:", df.isnull().sum().sum())
print("duplicate rows:", df.duplicated().sum())
df.describe().T.round(2)

## Step 1: Split first

Nothing is explored, scaled or selected before the split. Everything below looks at the
training data only, and the test set is opened once, at the end.

In [ ]:
X = df.drop(columns="target")
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RSEED
)

print(f"{X_train.shape[0]} training rows, {X_test.shape[0]} test rows")

## Step 2: Explore the training data

With a continuous target the first question is its distribution, not its class balance.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

sns.histplot(y_train, bins=25, ax=ax1)
ax1.set_title("Distribution of disease progression")
ax1.set_xlabel("progression score")

X_train.corrwith(y_train).sort_values().plot.barh(ax=ax2)
ax2.set_title("Correlation of each feature with the target");

The target is right-skewed and spans roughly 25 to 346, so an error of 50 is large. `bmi`
(0.61) and `s5`, triglycerides (0.55), carry the strongest linear relationship with
progression.

**`s3` is the only negative correlation, at -0.38, and that is the one to check.** `s3` is
HDL, the "good" cholesterol, so more of it going with less disease progression is exactly
what a clinician would predict. When the data agrees with domain knowledge about a column, that is
evidence that the data has been loaded and prepared correctly. When it disagrees, either the
data contains an unexpected finding or there is an error in the preparation, and an error is
the more frequent explanation.

The features are on very different scales, from `sex` (1 or 2) to `s1` (in the hundreds).
The penalty of Ridge and ElasticNet is only applied equally to all coefficients when the
features are on the same scale, so every pipeline below standardises the features first.

In [ ]:
X_train.describe().loc[["mean", "std", "min", "max"]].round(1)

## Step 3: A baseline

Before any model, decide what counts as good. `DummyRegressor` always predicts the mean of
the training target. Every model has to perform better than this baseline to be useful.

In [ ]:
cv = KFold(5, shuffle=True, random_state=RSEED)

baseline = DummyRegressor(strategy="mean")
baseline_rmse = -cross_val_score(
    baseline, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
).mean()

print(f"Baseline RMSE (always predict the mean): {baseline_rmse:.2f}")

## Step 4: Several models, cross-validated

We compare four linear models, each with the scaler inside the pipeline so that every fold is scaled using its own training part only:

- plain linear regression,
- Ridge and ElasticNet with their default settings,
- linear regression on **degree-2 polynomial features**, as in the lesson: every square and product of two features, 65 features in total.

In [ ]:
def scaled(model):
    """Standardise the features, then fit the model."""
    return Pipeline([("scaler", StandardScaler()), ("model", model)])


def poly_scaled(model):
    """Expand to degree-2 polynomial features, standardise them, then fit the model."""
    return Pipeline([
        ("poly", PolynomialFeatures(degree=2, include_bias=False)),
        ("scaler", StandardScaler()),
        ("model", model),
    ])


models = {
    "LinearRegression": scaled(LinearRegression()),
    "Ridge (untuned)": scaled(Ridge()),
    "ElasticNet (untuned)": scaled(ElasticNet()),
    "Polynomial + LinearRegression": poly_scaled(LinearRegression()),
}

for name, pipe in models.items():
    scores = -cross_val_score(
        pipe, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
    )
    print(
        f"{name:30} RMSE {scores.mean():6.2f}  (fold std {scores.std():4.2f}, "
        f"{baseline_rmse - scores.mean():+.2f} against baseline)"
    )

Three observations:

- **The three linear models on the ten original features beat the baseline by about 22 points** and are almost identical to each other (55.4 to 56.7). With ten features and 353 training rows, a linear model has little room to overfit, so a penalty has little to correct.
- **Untuned ElasticNet is slightly worse** than plain linear regression. Its default `alpha = 1.0` shrinks the coefficients more than this data requires.
- **Linear regression on the 65 polynomial features overfits.** Its RMSE rises to 62.5 and its folds vary much more (standard deviation 4.4 instead of 2.4): the same pattern as in the lesson, where unregularised linear regression on polynomial features gave very uneven folds.

The last result raises the question for the next step: can a tuned penalty make the polynomial model competitive?

## Step 5: Tune

We tune ElasticNet on the polynomial features, following the pattern from the lesson: first a wide randomized search to find a promising region, then a narrow grid search within it.

As in the lesson, the solver needs a looser tolerance (`tol=0.01`) and more iterations, and the smallest `alpha` values are excluded. `sex` only takes the values 1 and 2, so `sex²` is an exact linear function of `sex`, and with a very small penalty the solver does not converge on these perfectly correlated columns.

In [ ]:
elastic_net_pipe = poly_scaled(ElasticNet(tol=0.01, max_iter=10_000))

# Wide search: alpha over three orders of magnitude, the full range of l1_ratio
wide_distributions = {
    "model__alpha": loguniform(0.03, 30),
    "model__l1_ratio": uniform(0, 1),
}

rs = RandomizedSearchCV(
    elastic_net_pipe, wide_distributions, n_iter=30, cv=cv,
    scoring="neg_root_mean_squared_error", random_state=RSEED, n_jobs=-1,
)
rs.fit(X_train, y_train)

print("Random search best:", {k: round(float(v), 3) for k, v in rs.best_params_.items()})
print(f"Random search RMSE: {-rs.best_score_:.2f}")

In [ ]:
# Narrow grid around the region the random search found
narrow_grid = {
    "model__alpha": [0.03, 0.05, 0.08, 0.12],
    "model__l1_ratio": [0.1, 0.2, 0.3, 0.5],
}

gs = GridSearchCV(
    elastic_net_pipe, narrow_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
gs.fit(X_train, y_train)

print("Grid search best:", gs.best_params_)
print(f"Grid search RMSE: {-gs.best_score_:.2f}")

In [ ]:
# Ridge on the original ten features has one hyperparameter worth tuning: the penalty strength
ridge_pipe = scaled(Ridge())
ridge_grid = {"model__alpha": np.logspace(-3, 3, 25)}

ridge_gs = GridSearchCV(
    ridge_pipe, ridge_grid, cv=cv, scoring="neg_root_mean_squared_error", n_jobs=-1
)
ridge_gs.fit(X_train, y_train)

print("Best alpha:", round(ridge_gs.best_params_["model__alpha"], 4))
print(f"Ridge RMSE: {-ridge_gs.best_score_:.2f}")

Tuning brought the polynomial model from 62.5 down to about 55.2: a small `alpha` with a low `l1_ratio`, so mostly the Ridge part of the penalty, removes the overfitting. Tuning left Ridge on the original features almost unchanged (55.4), because a model with ten features has little to regularise.

**The penalty rescued the polynomial model, but it did not make it better than the simple one.** On the California housing data in the lesson, the polynomial features added information about location. Here they add none: the tuned polynomial model and plain linear regression are practically tied.

## Step 6: Choose a model, then use the test set once

The three candidates differ by about 0.2 in cross-validated RMSE, while the RMSE of a single model varies by more than 2 from fold to fold. A difference that is ten times smaller than the variation between folds is not a real difference.

When the scores are this close, choose the **simplest** model. A common rule is to take the simplest model whose score lies within one standard deviation of the best one (see the scikit-learn example [Balance model complexity and cross-validated score](https://scikit-learn.org/stable/auto_examples/model_selection/plot_grid_search_refit_callable.html)). Here that is plain linear regression: ten coefficients, no hyperparameters, and coefficients you can interpret, against 65 coefficients for the polynomial model.

In [ ]:
candidates = {
    "LinearRegression": models["LinearRegression"],
    "Ridge (tuned)": ridge_gs.best_estimator_,
    "Polynomial + ElasticNet (tuned)": gs.best_estimator_,
}

for name, model in candidates.items():
    scores = -cross_val_score(
        model, X_train, y_train, cv=cv, scoring="neg_root_mean_squared_error"
    )
    print(f"{name:32} RMSE {scores.mean():6.2f}  (fold std {scores.std():4.2f})")

In [ ]:
# The simplest of the tied models, fitted on the whole training set
best_name = "LinearRegression"
best_model = candidates[best_name].fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print(f"Chosen model: {best_name}\n")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred):7.2f}")
print(f"MAE:  {mean_absolute_error(y_test, y_pred):7.2f}")
print(f"R2:   {r2_score(y_test, y_pred):7.3f}")
print(f"\nBaseline RMSE for comparison: {baseline_rmse:.2f}")

**Reading these numbers in the units a doctor would use.** The progression score runs from
25 to 346. An MAE of about 43 means that a typical prediction differs from the true value by
43 points, about 13% of the whole range. The model is useful for identifying which patients
are likely to deteriorate, but not precise enough to predict the course for an individual
patient.

RMSE is larger than MAE because it squares the errors before averaging, so it is dominated
by the worst predictions. Reporting both gives a complete picture: MAE describes the typical
error, RMSE gives more weight to the largest errors.

An R2 of about 0.45 means that the model explains slightly less than half of the variation
in progression. The remaining variation depends on factors that are not in the dataset.

## Step 7: Look at the errors, not just the score

In [ ]:
residuals = y_test - y_pred

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.scatter(y_pred, residuals, alpha=0.6)
ax1.axhline(0, color="red", linestyle="--")
ax1.set_xlabel("predicted progression")
ax1.set_ylabel("residual (actual - predicted)")
ax1.set_title("Residuals against predictions")

ax2.scatter(y_test, y_pred, alpha=0.6)
lims = [y_test.min(), y_test.max()]
ax2.plot(lims, lims, color="red", linestyle="--")
ax2.set_xlabel("actual")
ax2.set_ylabel("predicted")
ax2.set_title("Predicted against actual");

The residuals are not random, and their pattern is important. If you split the test set
into quartiles of true progression, the mean residual is **-39, -14, +21, +51**: the
model **over-predicts the mildest cases and under-predicts the worst ones**, pulling every
prediction towards the average. On the right-hand plot the points cross the diagonal, rather
than scattering around it.

This is regression to the mean, and it is the type of error that matters most clinically:
the model makes its largest errors for the patients who deteriorate most, and it
underestimates their progression.

A single RMSE does not show this pattern. This is why the residual plots from Monday are
used again here, and why the last step of every workflow is to examine the errors, not only
the score.

## Summary of the workflow

1. Split before anything else.
2. Explore the training data only.
3. Establish a baseline, so "good" means something.
4. Compare several models under cross-validation, scaled inside a pipeline.
5. Tune wide, then narrow.
6. When cross-validated scores are within the variation between folds, choose the simplest model.
7. Use the test set once, for the chosen model.
8. Examine the errors, not only the score.